In [1]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
FEATURE_ROOT = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_109sub_full64"

METRICS_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/results/metrics"

os.makedirs(METRICS_DIR, exist_ok=True)

In [3]:
ALL_SUBJECTS = [
    f"S{i:03d}"
    for i in range(1,110)
]

In [4]:
EXPERIMENTS = {

    10: {
        "train": ALL_SUBJECTS[:8],
        "test": ALL_SUBJECTS[8:10]
    },

    20: {
        "train": ALL_SUBJECTS[:17],
        "test": ALL_SUBJECTS[17:20]
    },

    50: {
        "train": ALL_SUBJECTS[:42],
        "test": ALL_SUBJECTS[42:50]
    },

    109: {
        "train": ALL_SUBJECTS[:92],
        "test": ALL_SUBJECTS[92:109]
    }
}

In [5]:
def load_subject(subject):

    subject_dir = os.path.join(
        FEATURE_ROOT,
        subject
    )

    X = np.load(
        os.path.join(subject_dir, "X.npy")
    )

    y = np.load(
        os.path.join(subject_dir, "y.npy")
    )

    return X, y

In [6]:
all_results = []

In [7]:
for n_subjects, split in EXPERIMENTS.items():

    print("\n" + "="*60)
    print(f"Experiment : {n_subjects} Subjects")
    print("="*60)

    train_subjects = split["train"]
    test_subjects = split["test"]

    X_train = []
    y_train = []

    X_test = []
    y_test = []

    # -------------------------
    # TRAIN SET
    # -------------------------

    for subject in train_subjects:

        X, y = load_subject(subject)

        X_train.append(X)
        y_train.append(y)

    # -------------------------
    # TEST SET
    # -------------------------

    subject_accuracies = {}

    for subject in test_subjects:

        X, y = load_subject(subject)

        X_test.append(X)
        y_test.append(y)

    X_train = np.vstack(X_train)
    y_train = np.concatenate(y_train)

    X_test = np.vstack(X_test)
    y_test = np.concatenate(y_test)

    lda = LinearDiscriminantAnalysis()

    lda.fit(
        X_train,
        y_train
    )

    y_pred = lda.predict(X_test)

    overall_acc = accuracy_score(
        y_test,
        y_pred
    )

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    print(
        f"Overall Accuracy: "
        f"{overall_acc:.4f}"
    )

    # -------------------------
    # SUBJECT ACCURACIES
    # -------------------------

    for subject in test_subjects:

        X_sub, y_sub = load_subject(subject)

        pred_sub = lda.predict(X_sub)

        acc_sub = accuracy_score(
            y_sub,
            pred_sub
        )

        subject_accuracies[subject] = float(acc_sub)

        print(
            f"{subject}: "
            f"{acc_sub:.4f}"
        )

    result = {

        "experiment": f"{n_subjects}_subjects",

        "train_subjects": train_subjects,

        "test_subjects": test_subjects,

        "overall_accuracy": float(
            overall_acc
        ),

        "subject_accuracies":
            subject_accuracies,

        "confusion_matrix":
            cm.tolist(),

        "classification_report":
            classification_report(
                y_test,
                y_pred,
                output_dict=True
            )
    }

    save_path = os.path.join(
        METRICS_DIR,
        f"lda_{n_subjects}_subjects.json"
    )

    with open(
        save_path,
        "w"
    ) as f:

        json.dump(
            result,
            f,
            indent=4
        )

    all_results.append({

        "subjects":
            n_subjects,

        "accuracy":
            overall_acc
    })


Experiment : 10 Subjects
Overall Accuracy: 0.3778
S009: 0.3333
S010: 0.4222

Experiment : 20 Subjects
Overall Accuracy: 0.4778
S018: 0.5000
S019: 0.4333
S020: 0.5000

Experiment : 50 Subjects
Overall Accuracy: 0.5194
S043: 0.5333
S044: 0.5444
S045: 0.5222
S046: 0.5333
S047: 0.5000
S048: 0.5000
S049: 0.5222
S050: 0.5000

Experiment : 109 Subjects
Overall Accuracy: 0.4973
S093: 0.5000
S094: 0.5778
S095: 0.4778
S096: 0.4556
S097: 0.5556
S098: 0.5000
S099: 0.4556
S100: 0.3889
S101: 0.5000
S102: 0.5000
S103: 0.5000
S104: 0.5116
S105: 0.5000
S106: 0.5556
S107: 0.4667
S108: 0.4868
S109: 0.5000


In [8]:
summary_df = pd.DataFrame(
    all_results
)

summary_df

,subjects,accuracy
0,10,0.377778
1,20,0.477778
2,50,0.519444
3,109,0.497323
